<a href="https://colab.research.google.com/github/mochamadedwin/datascience/blob/main/pertemuan12_mochamadedwin_240401010293.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Aktivitas Hands-on pertemuan 10

Nama: Mochamad Edwin Nur Ishak

NIM: 240401010293

Kelas: IF - 403

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Daftar produk
produk = [
    'Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
    'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'
]

# Membuat 50 transaksi
# Setiap transaksi berisi 2-5 produk
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

# Menyuntikkan pola:
# Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

# Menampilkan hasil
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [4]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [18]:
from mlxtend.frequent_patterns import apriori
import warnings


warnings.filterwarnings("ignore", category=DeprecationWarning)
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(
        df,
        min_support=ms,
        use_colnames=True
    )

    print(
        f'min_support={ms}: {len(freq)} itemset ditemukan'
    )

# Gunakan min_support yang menghasilkan jumlah itemset wajar
freq_items = apriori(
    df,
    min_support=0.1,
    use_colnames=True
)

freq_items = freq_items.sort_values(
    'support',
    ascending=False
)

print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [17]:
from mlxtend.frequent_patterns import association_rules
import warnings


warnings.filterwarnings("ignore", category=DeprecationWarning)
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))

         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# Membuat katalog produk
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': [
        'Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
        'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy'
    ]
})

# Mengubah kategori menjadi fitur numerik
fitur = pd.get_dummies(katalog['kategori'])

# Menghitung cosine similarity antarproduk
sim_matrix = cosine_similarity(fitur)

# Fungsi rekomendasi produk yang memiliki kategori serupa
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[
        katalog['produk'] == nama_produk
    ][0]

    skor = list(enumerate(sim_matrix[idx]))

    # Mengurutkan dari similarity tertinggi
    skor = sorted(
        skor,
        key=lambda x: x[1],
        reverse=True
    )

    # Menghapus produk itu sendiri
    skor = [
        s for s in skor
        if s[0] != idx
    ][:top_n]

    return katalog.iloc[
        [i for i, _ in skor]
    ]['produk'].tolist()

print(
    'Mirip dengan Roti:',
    rekomendasi_serupa('Roti')
)


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


### Interpretasi Association Rules

Berdasarkan hasil Association Rule, aturan dengan nilai Lift tertinggi merupakan aturan yang memiliki hubungan paling kuat antara produk yang dibeli. Nilai Lift lebih dari 1 menunjukkan bahwa kedua produk memiliki hubungan positif, sehingga pembelian produk pada antecedent meningkatkan kemungkinan pembelian produk pada consequent.

Dari sisi bisnis, aturan seperti **Roti → Selai** masuk akal karena kedua produk dapat dikonsumsi secara bersamaan. Jika aturan tersebut memiliki nilai confidence dan lift yang tinggi, perusahaan dapat memanfaatkannya untuk membuat strategi bundling, memberikan rekomendasi produk, atau menempatkan kedua produk secara berdekatan.

In [25]:
produk_target = 'Roti'

# Dari association rules:
# mencari consequents dari aturan yang antecedent-nya mengandung produk target
rules_terkait = rules[
    rules['antecedents'].apply(
        lambda x: produk_target in x
    )
]

print('Rekomendasi dari Association Rules:')
print(
    rules_terkait[
        ['consequents', 'lift']
    ].head()
)

print(
    'Rekomendasi dari Content-Based:',
    rekomendasi_serupa(produk_target)
)

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Kesimpulan
Praktikum ini menunjukkan bahwa Apriori dan Association Rules dapat digunakan untuk menemukan pola hubungan antarproduk berdasarkan riwayat transaksi, sedangkan Cosine Similarity dapat digunakan untuk memberikan rekomendasi berdasarkan kemiripan kategori produk. Association Rules lebih berfokus pada pola pembelian pelanggan, sementara Content-Based Recommendation berfokus pada karakteristik produk. Dengan menggabungkan kedua pendekatan tersebut, sistem dapat menghasilkan rekomendasi yang lebih fleksibel dengan mempertimbangkan pola pembelian sekaligus kemiripan produk.